In [1]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

import requests

In [2]:
from functions import read_api_key

In [4]:
# Read data from previous work on cleaning the data
yerevan_historical = pd.read_csv('../database/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../database/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [5]:
# Dropping unnecessary columns and adding column for merging
yerevan_actual['backup_status'] = 'not sold'
yerevan_historical.drop(columns=[ 'days_in'], inplace=True)

In [6]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [7]:
# Dropping unnecessary columns
df.drop(columns=['links', 'source', 'rent_or_sale'], inplace=True)

### Removing houses that were wrongly classified as sold

In [9]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [10]:
df = df.dropna(subset=['district']).reset_index(drop=True)

### Categorizing Neighborhoods

as our data has x, y coordinates setup we wrote a script to generate the corresponding neighborhood for each house


this is later used for creating more interesting visuals in arcgis, and running the model with the context of the neighborhoods for each house

In [55]:
df2 = df.head(75732).reset_index(drop=True)

In [15]:
# function defined in functions.py
api_key = read_api_key()

addresses = []

In [ ]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            # send a request to the Google Maps API and get most specific 'APPROXIMATE' response
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

In [87]:
df['neighborhood'] = addresses

took postal code for a few of the houses, so we replaced the values to their corresponding neighborhoods

In [ ]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [87]:
# writing to csv
df.to_csv(r'../database/yerevan_combined_clean/sale_combined_neighborhood.csv', index=False)

small amount of rows weren't classified, so we remove them

In [88]:
df = pd.read_csv('../database/yerevan_combined_clean/sale_combined_neighborhood.csv')

In [182]:
df = df.dropna(subset=['neighborhood'])

In [184]:
# separating the dataframes from historical and actual
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

we have to remove neighborhoods that have a count lower than so that when we run models

that is only in the context of each neighborhood we don't get errors

In [186]:
# removing neighborhoods with low counts for historical
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [187]:
# removing neighborhoods with low counts for actual
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [196]:
# not needed for sale_actual as all nans
sale_actual.drop(columns=['sold_date'], inplace=True)

In [199]:
# Only taking date for sold_date
sale_historical['sold_date'] = pd.to_datetime(sale_historical['sold_date'])

# Extract the date part from the datetime column
sale_historical['sold_date'] = sale_historical['sold_date'].dt.date

In [200]:
# populate to csv
sale_historical.to_csv(r'../database/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../database/yerevan_combined_clean/actual_not_sold.csv', index=False)